# ДЗ1. Классификация текстов

В этой работе мы попробуем построить **спам-фильтр** для музыкальных видео. Обычно в комментариях к музыкальному видео вперемешку живут обсуждение песни, реклама чужих каналов, ссылки и просьбы подписаться. Мы попробуем сделать фильтр, который можно переносить на новое видео.

Исходный классификатор уже есть. Ваша задача сейчас - выяснить, какую информацию ему стоит оставить, придумать собственные признаки и проверить, как он реагирует на изменения в тексте.

Работа оценивается в **10 баллов**, распределение баллов отражено в таблице:

| Задание | Баллы |
|---|---:|
| 1. Подготовка данных и разметка | 2 |
| 2. Обработка под задачу | 2 |
| 3. Собственные признаки | 2 |
| 4. Что дает каждый набор признаков | 1.5 |
| 5. Проверки на изменённых текстах | 1.5 |
| 6. Новое видео | 1 |

По ходу оставляйте короткие ответы на вопросы. Корректный эксперимент, который не улучшил качество, оценивается так же, как эксперимент с улучшением. Фиксированного порога метрики для получения баллов нет.

**Формат сдачи**

Задание сдается через систему LMS. Посылка будет содержать:

- ноутбук `homework-01-Username.ipynb`, где Username - ваша фамилия латиницей без пробелов (например, `homework-01-Ivanov.ipynb`);
- файл `test_predictions.csv`.

Все таблицы, графики и выводы оставьте в тетрадке. Перед сдачей перезапустите ядро и выполните ячейки по порядку, чтобы проверить воспроизводимость результата.

> Источники и помощь генеративных моделей укажите в конце, но предполагается, что ключевые решения нужно уметь объяснить, поэтому размышления строго рекомендуется формулировать без LLM. По усмотрению ассистента при изобилии сгенерированного текста и кода балл может быть снижен.

## Подготовка

В работе мы будем использовать [YouTube Spam Collection](https://archive.ics.uci.edu/dataset/380/youtube+spam+collection) - это реальные комментарии к пяти музыкальным видео. Метка **1** обозначает спам, **0** - обычный комментарий. Это небольшой корпус 2015 года, на нём как раз будет удобно проверять идеи.

В исходном варианте 1956 комментариев.

Файлы лежат рядом с тетрадкой в `data/`. В Colab можно использовать ссылки из следующей ячейки или загрузить эту папку рядом с ноутбуком.

In [ ]:
%pip install -q "pandas==2.3.3" "scikit-learn==1.6.1" "matplotlib==3.9.4"

In [ ]:
from pathlib import Path
import html
import re
import platform
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, ConfusionMatrixDisplay

SEED = 13
pd.set_option('display.max_colwidth', 200)
plt.rcParams.update({'figure.figsize': (8, 4), 'font.size': 11})

In [ ]:
DATA_URL = (
    'https://raw.githubusercontent.com/AlbinaBurlova/hse-bda-nlp-course/'
    '2026/homeworks/HW1_text_classification/data/'
)
data_path = Path('data')

files = {
    'psy': 'Youtube01-Psy.csv',
    'katyperry': 'Youtube02-KatyPerry.csv',
    'lmfao': 'Youtube03-LMFAO.csv',
    'eminem': 'Youtube04-Eminem.csv',
    'shakira': 'Youtube05-Shakira.csv',
}
paths = {
    source: data_path / filename if (data_path / filename).exists() else DATA_URL + filename
    for source, filename in files.items()
}

**О задаче**

В исходных файлах `CONTENT` - текст, `CLASS` - метка, `COMMENT_ID` - идентификатор комментария, `AUTHOR` - имя автора, `DATE` - дата. Видео определяется файлом.

Мы проверяем перенос **на другое видео**. Разбиение фиксируем: трейн - Psy, Katy Perry и LMFAO; валидация - Eminem; тест - Shakira.

Схему, пропуски и повторы можно проверять во всей выгрузке. Идеи обработки и признаков берём из тренировочной выборки, модели сравниваем на валидации, тест используем для итоговой оценки. Модель получает только текст, метаданные не входят в признаки.

Основная метрика - **F1 спама, класса 1**. Рядом желательно показать precision и recall, ведь ошибочно скрытый обычный комментарий и пропущенная реклама - разные ошибки.

## 1. Подготовка данных и разметка [2 балла]

### 1.1. От файлов к таблице

Загрузите пять CSV через `pd.read_csv` и объедините в порядке из `files`. Добавьте `source` и присвойте строкам `id` от 0 до удаления каких-либо записей.

Переименуйте `CONTENT` в `text`, `CLASS` в `label` - это чтобы выдерживать нужный нейминг. Исходный текст тоже сохраните.

Проверьте состав данных по источникам и классам, пропуски и пустые тексты. Покажите, какие данные потеряются при удалении всех строк с пропусками. Оправданы ли эти потери для нашей задачи?

In [ ]:
# your code here
# (ﾉ◕ヮ◕)ﾉ*:･ﾟ✧

**Ответ:** # your text here (ಠ.ಠ)

### 1.2. Повторы и разбиение

Найдите точные повторы и проверьте - не скрываются ли одинаковые сообщения за разным оформлением? Выберите правила сравнения и создайте отдельный ключ, не меняя при этом исходный `text`. Обоснуйте свой выбор на нескольких группах из трейна.

> Для невидимых различий может пригодиться `repr()`

Решите, что делать с пропусками, пустыми текстами, повторами и возможными конфликтами меток. Объясните, какие строки сохраняете или исключаете.

Подготовьте трейн, валидацию и тест по заданному разбиению видео. В таблицах должны остаться `id`, `source`, `text`, `label` и `split`, порядок сохраняем по `id`.

Проверьте, что ID уникальны, в каждой части есть оба класса, а видео и одинаковые тексты не пересекаются между частями. Для текстов проверьте и точное совпадение, и ваш ключ. Если группа попала в разные части, выберите правило исключения записей. Повторы внутри части можно оставить с обоснованием.

Покажите размеры и состав классов до и после подготовки. Единого правильного числа строк нет. Зафиксируйте выборку: все следующие варианты модели сравниваем на одних и тех же строках, без удаления примеров по результатам предсказаний.

In [ ]:
# your code here
# (⌐■_■)

**Ответ:** # your text here (ಠ.ಠ)

### 1.3. Что считать спамом

Сформулируйте своё правило: какие комментарии вы бы скрывали как спам?

Выберите **20 случайных комментариев из train** и поставьте свои метки: 0, 1 или «сомневаюсь», не глядя на метки корпуса. Затем сравните ответы и сохраните таблицу.

Давайте обсудим расхождения/пограничные случаи. Предложите конкретное уточнение инструкции разметчикам. Здесь оценивается именно обоснование. Но в обучении продолжаем использовать исходные метки.

**Баллы за задание 1:** 1 за подготовку и разбор проблем, 0.5 за проверки разбиения, 0.5 за собственную разметку и уточнение инструкции.

**Правило спама до просмотра исходных меток:**

**Ответ:** # your text here (ಠ.ಠ)

In [ ]:
# your code here
# (づ｡◕‿‿◕｡)づ
# Выбор комментариев и собственные метки.

In [ ]:
# your code here
# (⌐■_■)
# Сравнение с исходными метками.

**Сравнение меток и уточнение инструкции:**

**Ответ:** # your text here (ಠ.ಠ)

## Стартовая точка

Данные готовы. Начнём со знакомого словного TF-IDF и логистической регрессии.

В `make_model` можно передать `preprocessor` - вашу функцию «строка -> строка», и `feature_function` - функцию, которая получает исходные тексты и возвращает матрицу «число текстов × число признаков». `FeatureUnion` объединяет представления, масштабирование числовых признаков выполняется внутри `Pipeline`.

> Собственный `preprocessor` заменяет стандартную предобработку, включая перевод в нижний регистр. Решение о регистре нужно учесть в своей функции.

In [ ]:
def make_model(preprocessor=None,feature_function=None,use_words=True,C=1):
    branches=[]
    if use_words:
        branches.append(('words',TfidfVectorizer(
            min_df=2,max_features=10000,preprocessor=preprocessor,
            lowercase=preprocessor is None,
        )))
    if feature_function is not None:
        branches.append(('signals',Pipeline([
            ('extract',FunctionTransformer(feature_function,validate=False)),
            ('scale',StandardScaler()),
        ])))
    if not branches:
        raise ValueError('Нужен хотя бы один набор признаков')
    return Pipeline([
        ('features',FeatureUnion(branches)),
        ('classifier',LogisticRegression(C=C,max_iter=1000,random_state=13)),
    ])

In [ ]:
def quality(y_true, y_pred):
    return {
        'F1 spam': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision spam': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall spam': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

baseline = make_model()
baseline.fit(train['text'], train['label'])
baseline_prediction = baseline.predict(valid['text'])
display(pd.DataFrame([quality(valid['label'], baseline_prediction)], index=['TF-IDF']))

## 2. Обработка под задачу [2 балла]

Изучите train и выберите обработку, которая имеет смысл для этих комментариев. Реализуйте функцию «строка -> строка». Набор операций и инструменты выбираете сами.

Покажите действие функции на нескольких настоящих комментариях с разными особенностями. Что изменилось, какая информация сохранилась, а какая потерялась? Есть ли риск убрать полезный для классификации сигнал?

Проверьте повторное применение функции и появление пустых результатов. Объясните обнаруженные проблемы, если они есть. Строки из подготовленных выборок не удаляем.

**Баллы:** 1 за обоснование и реализацию обработки, 0.5 за примеры и анализ изменений, 0.5 за проверки.

In [ ]:
# your code here
# ٩(⁎❛ᴗ❛⁎)۶

In [ ]:
# your code here
# ヾ(๑╹◡╹)ﾉ

**Ответ:** # your text here (ಠ.ಠ)

## 3. Собственные признаки [2 балла]

Придумайте числовые или бинарные признаки, которые могут помочь отличить спам от обычных комментариев на YouTube. Каждый признак должен вычисляться по тексту комментария, по одному и тому же правилу для всех сообщений.

Для каждого признака объясните, что он измеряет и почему вы ожидаете, что он будет полезен. Набор и количество признаков выбирайте самостоятельно.

-> Реализуйте функцию, возвращающую матрицу признаков. Проверьте её размер и отсутствие пропусков или бесконечностей.

На трейне сравните распределения или сводную статистику признаков по классам. Подтвердились ли ваши предположения? Найдите настоящий комментарий, показывающий ограничение хотя бы одного признака.

**Баллы:** 1 за признаки с обоснованием, 0.5 за реализацию и проверки, 0.5 за статистику и контрпример.

In [ ]:
# your code here
# ฅ^•ﻌ•^ฅ

In [ ]:
# your code here
# ʕ•ᴥ•ʔ

**Ответ:** # your text here (ಠ.ಠ)

## 4. Что дает каждый набор признаков [1.5 балла]

### 4.1. Общий результат

Сравните на валидации четыре варианта:

| Вариант | Вход классификатора |
|---|---|
| Исходный | Готовый TF-IDF без вашей обработки |
| Обработка | TF-IDF после вашей функции |
| Собственные признаки | Только признаки из задания 3 |
| Вместе | TF-IDF после обработки + признаки из задания 3 |

Используйте одинаковые настройки логистической регрессии, включая `C`, и одинаковые параметры TF-IDF. Соберите таблицу с F1, precision и recall спама.

Какой вклад дали обработка и дополнительные признаки? Разберите два настоящих комментария из валидации, которые помогают объяснить сравнение моделей. Улучшение метрики при этом показывать не обязательно.

In [ ]:
# your code here
# (っ˘ڡ˘ς)

**Общий результат и разбор двух комментариев**

**Ответ:** # your text here (ಠ.ಠ)

### 4.2. Отдельный тип комментариев

По наблюдениям на трейне выберите один тип комментариев и задайте правило отбора, не зависящее от прогнозов модели. На этом срезе валидации сравните исходную модель с одной из доработанных: покажите метрики и число объектов каждого класса.

*Сохраняется ли вывод по общей метрике?*

Выберите кандидата для финальной проверки с учётом общего результата и среза. Объясните решение.

**Баллы за задание 4:** 0.5 за корректное сравнение четырёх вариантов, 0.5 за проверку и интерпретацию среза, 0.5 за разбор примеров и обоснованный выбор.

In [ ]:
# your code here
# (•̀ᴗ•́)و

**Результат на срезе и выбор модели:**

**Ответ:** # your text here (ಠ.ಠ)

## 5. Проверки на измененных текстах [1.5 балла]

Выберите **8 настоящих комментариев из валидации**, лучше оба класса. Придумайте **два преобразования**, которые сохраняют смысл и метку этих сообщений.

А получится хотя бы одно преобразование связать с риском вашей обработки/признаков? Объясните, что может измениться в представлении текста, доступном модели.

Без переобучения сравните исходную модель с одной из доработанных. Для каждой модели и преобразования посчитайте новые и исправленные ошибки.

Разберите какой-нибудь показательный пример. Если новых ошибок нет, объясните, чего эта проверка всё ещё не гарантирует?

**Баллы:** 0.5 за обоснование преобразований и сохранения метки, 0.5 за таблицу и подсчёты, 0.5 за разбор результата и связь с представлением текста.

In [ ]:
# your code here
# =^･ｪ･^=

**Ответ:** # your text here (ಠ.ಠ)

## 6. Новое видео [1 балл]

Зафиксируйте окончательный выбор до оценки на тесте - это наша последняя граница. Обучите весь выбранный `Pipeline` заново на train + validation, включая словарь, IDF и масштабирование, если они используются. Оцените его на тесте: F1, precision, recall спама и матрица ошибок.

Сохраните `test_predictions.csv` с колонками `id` и `prediction` для всех строк вашего test в порядке `id`. Мы этот файл будем прикреплять вместе с ноутбуком в lms.

Переносится ли решение на новое видео? Какие ошибки остаются и что нужно проверить перед реальным использованием?

**Баллы:** 0.5 за итоговую оценку и файл, 0.5 за обоснованный вывод и воспроизводимость.

In [ ]:
# your code here
# (⊙_⊙)

**Ответ:** # your text here (ಠ.ಠ)

## Источники и помощь

Укажите использованные материалы и помощь генеративных моделей. Если заимствовали код, отметьте источник и объясните, что изменили.

**Ответ:** # your text here (ಠ.ಠ)